<div class="alert alert-block alert-info">

### Imports & Connection
</div>

In [18]:
import os
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import datetime as datetime
import plotly.graph_objects as go

from scipy import stats
from pyhive import presto
from IPython.display import display
from datetime import datetime, timedelta
# from scipy.stats import gaussian_kde

import warnings
warnings.filterwarnings('ignore')

In [19]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

In [23]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        # 'presto-gateway.processing.data.production.internal',
        # 'presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [24]:
print('Notebook dir - ', os.getcwd())
local_extract = '/Users/E2074/local-datasets/customer/ads-monetisation/vivo-video-campaign'
print('Data dir - ', local_extract)

Notebook dir -  /Users/E2074/analytics/customer/Ads-Monetisation/vivo-video-campaign
Data dir -  /Users/E2074/local-datasets/customer/ads-monetisation/vivo-video-campaign


<div class="alert alert-block alert-info">
    
## Datasets & Parameter
</div>

In [25]:
## Parameter 
start_date = '20241121'
end_date = '20241126'

<div class="alert alert-block alert-warning">
    
#### Required Query Function 
</div>

In [26]:
def func_ad_response(date, formatted_end_date):
    
    base_query = f"""
    
        WITH response AS (
            SELECT
                yyyymmdd,
                userid,
                pagename,
                slotname,
                orderstatus,
                -- CASE WHEN slotname LIKE '%CaptainSearch%' THEN 'CaptainSearch1' ELSE orderstatus END AS orderstatus,
                -- CASE WHEN REGEXP_LIKE(slotname, 'CaptainSearch') = true THEN 'CaptainSearch1' ELSE orderstatus END AS orderstatus,
                format,
                service,
                REPLACE(REGEXP_EXTRACT(metadata1, 'title=([^,]*)'), 'title=', '') title,
                metadata1,
                coalesce(aduuid, adid) ad_response_id
                
            FROM 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE 
                yyyymmdd = '{date}'
                AND responsetype = 'GAMBanner'
                AND eventname = 'Ad_Response'
                -- AND REPLACE(REGEXP_EXTRACT(metadata1, 'title=([^,]*)'), 'title=', '') IN ('vivo Y300 5G', 'vivo Y300')
                AND metadata1 LIKE '%vivo Y300%'
                -- AND city = 'Bangalore'
                
        
            ),
            segment AS (
            
            SELECT 
                customer_id,
                taxi_ltr_segment,
                taxi_retention_segments,
                taxi_income_segment,
                taxi_service_affinity,
                taxi_regularity_segment,
                taxi_fe_regularity_segment
            FROM 
                datasets.iallocator_customer_segments 
            WHERE 
                run_date = '{formatted_end_date}'
                AND fe_recency_type IN ('Recent','Stationary','Dormant')
            )

            SELECT
                response.*,
                taxi_ltr_segment,
                taxi_retention_segments,
                taxi_income_segment,
                taxi_service_affinity,
                taxi_regularity_segment,
                taxi_fe_regularity_segment
            FROM 
                response
            LEFT JOIN
                segment
                ON response.userid = segment.customer_id
            

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [27]:
def func_ad_impression(date):
    
    base_query = f"""
    
            SELECT
                yyyymmdd,
                userid,
                coalesce(aduuid, adid) ad_impression_id
            FROM 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE 
                yyyymmdd = '{date}'
                AND responsetype = 'GAMBanner'
                AND eventname = 'Ad_Impression'
                -- AND city = 'Bangalore'

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [28]:
def func_ad_viewable_impression(date):
    
    base_query = f"""
    
            SELECT
                yyyymmdd,
                userid,
                coalesce(aduuid, adid) ad_viewable_impression_id
            FROM 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE 
                yyyymmdd = '{date}'
                AND responsetype = 'GAMBanner'
                AND eventname = 'Ad_Viewable_Impression'
                -- AND city = 'Bangalore'

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [29]:
def func_ad_click(date):
    
    base_query = f"""
    
            SELECT
                yyyymmdd,
                userid,
                coalesce(aduuid, adid) ad_click_id
            FROM 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            WHERE 
                yyyymmdd = '{date}'
                AND responsetype = 'GAMBanner'
                AND eventname = 'Ad_Click'
                -- AND city = 'Bangalore'

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [30]:
def func_pull_ads_data(start_date, end_date):
    
    date_list = pd.date_range(start=start_date, end=end_date).strftime('%Y%m%d').tolist()
    formatted_end_date = (datetime.strptime(start_date, '%Y%m%d') - timedelta(days=1)).strftime('%Y-%m-%d')

    
    for date in date_list:
            
        print(date)
        df_response = func_ad_response(date, formatted_end_date)
        print('response done')
        df_impression = func_ad_impression(date)
        print('impression done')
        df_viewable_impression = func_ad_viewable_impression(date)
        print('resviewable_impressionponse done')
        df_click = func_ad_click(date)
        print('click done')

        merge1 = pd.merge(df_response, df_impression, 
                          how='left', 
                          left_on=['yyyymmdd', 'userid', 'ad_response_id'],
                          right_on=['yyyymmdd', 'userid', 'ad_impression_id'])

        merge2 = pd.merge(merge1, df_viewable_impression, 
                          how='left', 
                          left_on=['yyyymmdd', 'userid', 'ad_impression_id'],
                          right_on=['yyyymmdd', 'userid', 'ad_viewable_impression_id'])

        merge3 = pd.merge(merge2, df_click, 
                          how='left', 
                          left_on=['yyyymmdd', 'userid', 'ad_viewable_impression_id'],
                          right_on=['yyyymmdd', 'userid', 'ad_click_id'])

        merge3.to_parquet(local_extract + '/data_dump/{}_data.parquet'.format(date), index=False)

In [32]:
func_pull_ads_data(start_date, end_date)

20241121


DatabaseError: {'message': 'Query exceeded the maximum execution time limit of 10.00m', 'errorCode': 131075, 'errorName': 'EXCEEDED_TIME_LIMIT', 'errorType': 'INSUFFICIENT_RESOURCES', 'failureInfo': {'type': 'io.trino.spi.TrinoException', 'message': 'Query exceeded the maximum execution time limit of 10.00m', 'suppressed': [], 'stack': ['io.trino.execution.QueryTracker.enforceTimeLimits(QueryTracker.java:187)', 'io.trino.execution.QueryTracker.lambda$start$0(QueryTracker.java:89)', 'java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:515)', 'java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)', 'java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)', 'java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1128)', 'java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:628)', 'java.base/java.lang.Thread.run(Thread.j

In [ ]:
folder_path = local_extract + '/data_dump'
parquet_files = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith('.parquet')]
df = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)

In [11]:
# df = func_pull_ads_data(start_date, end_date, city_list)

<div class="alert alert-block alert-warning">
    
#### User Defined Function & Required Variable 
</div>

In [26]:
pattern_format = r"OTW|Arrived|Started|nativeBanner|default|MAP_BRANDING"
pattern_os = r"iPadOS"

agg_cols = {
    'responses' : ('response_id', 'nunique'),
    'impressions' : ('impression_id', 'nunique'),
    'viewable_impressions' : ('viewable_impression_id', 'nunique'),
    'clicks' : ('click_id', 'nunique'),
}

In [36]:
def func_percentage(df, numerator, denominator):
    df['ctr'] = (df[numerator]*100.00/df[denominator]).round(2)
    return df

def group_aggregate_data(df, groupby_cols, agg_col, numerator, denominator):
    df_grouped = df.groupby(groupby_cols).agg(**agg_col).reset_index()
    df_grouped = func_percentage(df_grouped, numerator, denominator)
    df_grouped.replace([np.inf, -np.inf], 0.0, inplace=True)
    df_grouped.fillna(0.0, inplace=True)
    
    return df_grouped

<div class="alert alert-block alert-warning">
    
#### Reading Extracted data
</div>

In [28]:
prefix_list = pd.date_range(start=start_date, end=end_date).strftime('%Y%m%d').tolist()
cities_list = ['_Chennai_' ] # '_Bangalore_', '_Chennai_', '_Delhi_', '_Hyderabad_'

path_to_directory = local_extract + '/data_dump/'
dataframes = []

In [29]:
for prefix in prefix_list[0:-2]:
   
    print(f"Processing date: {prefix}")
    parquet_files = glob.glob(os.path.join(path_to_directory, f"{prefix}*.parquet"))
    filtered_cities = [file for file in parquet_files if any(city in file for city in cities_list)]

    for city in filtered_cities:
        dataframes.append(pd.read_parquet(city))
        print(f"Loaded city: {city}")
        

df_data = pd.concat(dataframes, ignore_index=True)
df_data = df_data[~df_data["format"].str.contains(pattern_format, regex=True)]
df_data = df_data[~df_data["os"].str.contains(pattern_os, regex=True)]

Processing date: 20241128
Loaded city: /Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark/data_dump/20241128_Chennai_data.parquet
Processing date: 20241129
Loaded city: /Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark/data_dump/20241129_Chennai_data.parquet
Processing date: 20241130
Loaded city: /Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark/data_dump/20241130_Chennai_data.parquet
Processing date: 20241201
Loaded city: /Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark/data_dump/20241201_Chennai_data.parquet
Processing date: 20241202
Loaded city: /Users/E2074/local-datasets/customer/ads-monetisation/monet-benchmark/data_dump/20241202_Chennai_data.parquet


In [30]:
df_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9290980 entries, 0 to 11695367
Data columns (total 19 columns):
 #   Column                      Dtype 
---  ------                      ----- 
 0   yyyymmdd                    object
 1   city                        object
 2   os                          object
 3   responsetype                object
 4   userid                      object
 5   slotname                    object
 6   format                      object
 7   responsestatus              object
 8   taxi_ltr_segment            object
 9   taxi_retention_segment      object
 10  taxi_rr_regularity_segment  object
 11  taxi_fe_regularity_segment  object
 12  response_id                 object
 13  title                       object
 14  advertiserName              object
 15  body                        object
 16  impression_id               object
 17  viewable_impression_id      object
 18  click_id                    object
dtypes: object(19)
memory usage: 1.4+ GB


In [31]:
df_data.shape

(9290980, 19)

In [32]:
df_data.head(1)

,yyyymmdd,city,os,responsetype,userid,slotname,format,responsestatus,taxi_ltr_segment,taxi_retention_segment,taxi_rr_regularity_segment,taxi_fe_regularity_segment,response_id,title,advertiserName,body,impression_id,viewable_impression_id,click_id
0,20241128,Chennai,iOS,GAMBanner,61b0989d17dd6a21550df47b,CaptainSearch1,gamCombinedBanner,true,PHH,SILVER,BI_WEEKLY,MONTHLY,61b0989d17dd6a21550df47b_05FE802F-24AA-4147-A8...,None,None,None,None,None,None


<div class="alert alert-block alert-success">  
    
## Analysis
</div>

In [33]:
df_data.columns

Index(['yyyymmdd', 'city', 'os', 'responsetype', 'userid', 'slotname',
       'format', 'responsestatus', 'taxi_ltr_segment',
       'taxi_retention_segment', 'taxi_rr_regularity_segment',
       'taxi_fe_regularity_segment', 'response_id', 'title', 'advertiserName',
       'body', 'impression_id', 'viewable_impression_id', 'click_id'],
      dtype='object')

In [34]:
df_data.head(3)

,yyyymmdd,city,os,responsetype,userid,slotname,format,responsestatus,taxi_ltr_segment,taxi_retention_segment,taxi_rr_regularity_segment,taxi_fe_regularity_segment,response_id,title,advertiserName,body,impression_id,viewable_impression_id,click_id
0,20241128,Chennai,iOS,GAMBanner,61b0989d17dd6a21550df47b,CaptainSearch1,gamCombinedBanner,true,PHH,SILVER,BI_WEEKLY,MONTHLY,61b0989d17dd6a21550df47b_05FE802F-24AA-4147-A8...,None,None,None,None,None,None
1,20241128,Chennai,iOS,GAMBanner,664edf309f5bb30c2f50ffba,PostOrderStarted4,gamCombinedBanner,true,PHH,PLATINUM,WEEKLY,WEEKLY,664edf309f5bb30c2f50ffba_1A6C52BB-CF5C-4254-9D...,None,None,None,None,None,None
2,20241128,Chennai,iOS,GAMBanner,634d566c051617dda2017732,PostOrderOnTheWay3,gamCombinedBanner,true,PHH,GOLD,WEEKLY,BI_WEEKLY,634d566c051617dda2017732_1E82B747-7C92-439D-8C...,None,None,None,634d566c051617dda2017732_1E82B747-7C92-439D-8C...,634d566c051617dda2017732_1E82B747-7C92-439D-8C...,None


<div class="alert alert-block alert-success">  
    
#### Overall Numbers
</div>

In [37]:
groupby_cols = ['os', 'responsetype', 'format']

group_aggregate_data(df_data, groupby_cols, agg_cols, 'clicks', 'viewable_impressions')

,os,responsetype,format,responses,impressions,viewable_impressions,clicks,ctr
0,android,GAMBanner,,52721,29245,24945,170,0.68
1,android,GAMBanner,gamCombinedBanner,7523106,1292672,1096890,5481,0.50
2,android,GAMBanner,nativeVideoBanner,22979,33,31,1,3.23
3,android,ONLINE_SALES,maxBanner,383,321,291,4,1.37
4,android,System Banner,Rewards,1108660,916342,666555,4949,0.74
5,iOS,GAMBanner,gamCombinedBanner,427120,178549,151435,1660,1.10
6,iOS,ONLINE_SALES,maxBanner,80,54,6,1,16.67


In [22]:
# source (GAM/OS/SB)
# os (AND/iOS)
# format (Image/Video/Story)

In [23]:
# taxi_ltr_segment (HH/PHH)
# taxi_retention_segment (SGPE)
# taxi_rr_regularity_segment (Daily/Weekly/Monthly)

In [24]:
# slotname(slot1/2/3)

In [25]:
# metadata-category